<a href="https://colab.research.google.com/github/ntatfff/todo/blob/202411241258/ColabRadiomicsFeatureExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
kernel = 1
className = 'firstorder'

In [2]:
# Parameters
kernel = 3
className = "firstorder"


In [3]:
# Utilities: loaders and validation
import os
import sys
import glob
import csv
from typing import List, Tuple
import numpy as np
import SimpleITK as sitk

def load_array(path: str) -> np.ndarray:
    ext = os.path.splitext(path)[1].lower()
    if ext != '.nrrd':
        raise ValueError(f"Unsupported file type for this notebook (expected .nrrd): {path}")
    image = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(image).astype(np.float32)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D array, got shape {arr.shape} for {path}")
    return arr

def validate_same_shape(arrs: List[np.ndarray]) -> Tuple[int, int, int]:
    shapes = [a.shape for a in arrs]
    if len(set(shapes)) != 1:
        raise ValueError(f"All arrays must have the same shape, got: {shapes}")
    return arrs[0].shape

In [ ]:
# Combine and export to CSV using pandas
import pandas as pd
import os
from pathlib import Path 

def nrrd2csv(input_files: List[str], output_csv: str):
  arrays = [load_array(p) for p in input_files]
  shape = validate_same_shape(arrays)
  N = len(arrays)

  # Stack as (N, X, Y, Z) then reshape to (voxels, N)
  stacked = np.stack(arrays, axis=0)  # (N, D, H, W)
  vox_mat = stacked.reshape(N, -1).T       # (D*H*W, N)

  # Create DataFrame and write CSV
  # Column names derived from final token before extension in filename
  # Example: original_firstorder_10Percentile.nrrd -> 10Percentile
  base_names = [os.path.splitext(os.path.basename(p))[0] for p in input_files]
  cols = [bn.split('_')[-1] if '_' in bn else bn for bn in base_names]
  df = pd.DataFrame(vox_mat, columns=cols)

  Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
  df.to_csv(output_csv, index=False)
  print(f"Saved CSV to: {output_csv} with columns: {cols}")

In [ ]:
import radiomics
import numpy as np
import SimpleITK as sitk
import radiomics.featureextractor
import os
import six
from os import path
import pandas as pd

def featureExtractor(patientId):
  imagePath = f"./dataset/BraTS2021_Training_Data/{patientId}/{patientId}_flair.nii.gz"
  image = sitk.ReadImage(imagePath)
  maskPath = f"./dataset/BraTS2021_Training_Data/{patientId}/{patientId}_kernel{kernel}_tumor.nii.gz"
  mask = sitk.ReadImage(maskPath)

  settings = {}
  settings['kernelRadius'] = kernel
  settings['maskedKernel'] = False
  settings['voxelBatch'] = 1000
  extractor = radiomics.featureextractor.RadiomicsFeatureExtractor(**settings)
  extractor.disableAllFeatures()
  extractor.enableFeatureClassByName(className)

  featureMap = extractor.execute(image, mask, voxelBased=True)

  patientFolder = f"./dataset/{className}/kernel{kernel}/tumor/{patientId}"
  for featureName, featureValue in six.iteritems(featureMap):
    if isinstance(featureValue, sitk.Image):
      if path.exists(patientFolder) == False:
        os.makedirs(patientFolder, exist_ok=True)
      sitk.WriteImage(featureValue, f"{patientFolder}/{featureName}.nrrd")
      print(f"Computed {featureName}, stored as {patientFolder}/{featureName}.nrrd")
    else:
      print(f"{featureName}: {featureValue}")

  # convert nrrd to csv
  patientPath = patientFolder
  featureFiles = list(filter(lambda f: f.endswith('.nrrd'), os.listdir(patientPath)))
  featureFiles.sort()
  featureFiles = [f"{patientPath}/{featureFile}" for featureFile in featureFiles]
  outputCsvPath = f"{patientPath}/nrrd2csv.csv"
  if os.path.exists(outputCsvPath):
    print(f"CSV already exists for {patientFolder}, skipping.")
    for featureFile in featureFiles:
      os.remove(featureFile)
  else:
    nrrd2csv(featureFiles, outputCsvPath)
    for featureFile in featureFiles:
      os.remove(featureFile)

In [6]:
# main
monitorFilePath = f"./dataset/{className}/kernel{kernel}/tumor.monitor.csv"
monitor = pd.read_csv(monitorFilePath, index_col='no')
for i, row in monitor.iterrows():
  if row['done'] != 0:
    continue
  patientId = row['file']
  print('Starting %s' % (patientId))
  featureExtractor(patientId)
  monitor.at[i, 'done'] = 1
  monitor.to_csv(monitorFilePath)

Starting BraTS2021_00000


diagnostics_Versions_PyRadiomics: v3.1.0
diagnostics_Versions_Numpy: 1.26.4
diagnostics_Versions_SimpleITK: 2.4.0
diagnostics_Versions_PyWavelet: 1.6.0
diagnostics_Versions_Python: 3.9.24
diagnostics_Configuration_Settings: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'kernelRadius': 3, 'maskedKernel': False, 'voxelBatch': 1000, 'voxelBased': True}
diagnostics_Configuration_EnabledImageTypes: {'Original': {}}
diagnostics_Image-original_Hash: 6e43038c244d0be8617af2e11f614fe457c2ef11
diagnostics_Image-original_Dimensionality: 3D
diagnostics_Image-original_Spacing: (1.0, 1.0, 1.0)
diagnostics_Image-original_Size: (240, 240, 155)
diagnostics_Image-original_Mean: 163.6575160170251
diagnostics_Image-original

Saved CSV to: ./dataset/firstorder/kernel3/tumor/BraTS2021_00000/nrrd2csv.csv with columns: ['10Percentile', '90Percentile', 'Energy', 'Entropy', 'InterquartileRange', 'Kurtosis', 'Maximum', 'Mean', 'MeanAbsoluteDeviation', 'Median', 'Minimum', 'Range', 'RobustMeanAbsoluteDeviation', 'RootMeanSquared', 'Skewness', 'TotalEnergy', 'Uniformity', 'Variance']
Starting BraTS2021_00002


diagnostics_Versions_PyRadiomics: v3.1.0
diagnostics_Versions_Numpy: 1.26.4
diagnostics_Versions_SimpleITK: 2.4.0
diagnostics_Versions_PyWavelet: 1.6.0
diagnostics_Versions_Python: 3.9.24
diagnostics_Configuration_Settings: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'kernelRadius': 3, 'maskedKernel': False, 'voxelBatch': 1000, 'voxelBased': True}
diagnostics_Configuration_EnabledImageTypes: {'Original': {}}
diagnostics_Image-original_Hash: 97c1ae5fa89c22c510aca9321071ab16c519cddc
diagnostics_Image-original_Dimensionality: 3D
diagnostics_Image-original_Spacing: (1.0, 1.0, 1.0)
diagnostics_Image-original_Size: (240, 240, 155)
diagnostics_Image-original_Mean: 134.25051993727598
diagnostics_Image-origina

Computed original_firstorder_Mean, stored as ./dataset/firstorder/kernel3/tumor/BraTS2021_00002/original_firstorder_Mean.nrrd
Computed original_firstorder_Median, stored as ./dataset/firstorder/kernel3/tumor/BraTS2021_00002/original_firstorder_Median.nrrd
Computed original_firstorder_Minimum, stored as ./dataset/firstorder/kernel3/tumor/BraTS2021_00002/original_firstorder_Minimum.nrrd
Computed original_firstorder_Range, stored as ./dataset/firstorder/kernel3/tumor/BraTS2021_00002/original_firstorder_Range.nrrd
Computed original_firstorder_RobustMeanAbsoluteDeviation, stored as ./dataset/firstorder/kernel3/tumor/BraTS2021_00002/original_firstorder_RobustMeanAbsoluteDeviation.nrrd
Computed original_firstorder_RootMeanSquared, stored as ./dataset/firstorder/kernel3/tumor/BraTS2021_00002/original_firstorder_RootMeanSquared.nrrd
Computed original_firstorder_Skewness, stored as ./dataset/firstorder/kernel3/tumor/BraTS2021_00002/original_firstorder_Skewness.nrrd
Computed original_firstorder_T

...

diagnostics_Versions_PyRadiomics: v3.1.0
diagnostics_Versions_Numpy: 1.26.4
diagnostics_Versions_SimpleITK: 2.4.0
diagnostics_Versions_PyWavelet: 1.6.0
diagnostics_Versions_Python: 3.9.24
diagnostics_Configuration_Settings: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'kernelRadius': 3, 'maskedKernel': False, 'voxelBatch': 1000, 'voxelBased': True}
diagnostics_Configuration_EnabledImageTypes: {'Original': {}}
diagnostics_Image-original_Hash: 7427cb8416e85e5d5b442e1fae18506e90cfafaf
diagnostics_Image-original_Dimensionality: 3D
diagnostics_Image-original_Spacing: (1.0, 1.0, 1.0)
diagnostics_Image-original_Size: (240, 240, 155)
diagnostics_Image-original_Mean: 151.46429289874553
diagnostics_Image-origina

Saved CSV to: ./dataset/firstorder/kernel3/tumor/BraTS2021_00014/nrrd2csv.csv with columns: ['10Percentile', '90Percentile', 'Energy', 'Entropy', 'InterquartileRange', 'Kurtosis', 'Maximum', 'Mean', 'MeanAbsoluteDeviation', 'Median', 'Minimum', 'Range', 'RobustMeanAbsoluteDeviation', 'RootMeanSquared', 'Skewness', 'TotalEnergy', 'Uniformity', 'Variance']
Starting BraTS2021_00016


diagnostics_Versions_PyRadiomics: v3.1.0
diagnostics_Versions_Numpy: 1.26.4
diagnostics_Versions_SimpleITK: 2.4.0
diagnostics_Versions_PyWavelet: 1.6.0
diagnostics_Versions_Python: 3.9.24
diagnostics_Configuration_Settings: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'kernelRadius': 3, 'maskedKernel': False, 'voxelBatch': 1000, 'voxelBased': True}
diagnostics_Configuration_EnabledImageTypes: {'Original': {}}
diagnostics_Image-original_Hash: 86875b7622c6a34cbff2d924dea72b069934d15f
diagnostics_Image-original_Dimensionality: 3D
diagnostics_Image-original_Spacing: (1.0, 1.0, 1.0)
diagnostics_Image-original_Size: (240, 240, 155)
diagnostics_Image-original_Mean: 123.67769914874552
diagnostics_Image-origina

Computed original_firstorder_Variance, stored as ./dataset/firstorder/kernel3/tumor/BraTS2021_00016/original_firstorder_Variance.nrrd


Saved CSV to: ./dataset/firstorder/kernel3/tumor/BraTS2021_00016/nrrd2csv.csv with columns: ['10Percentile', '90Percentile', 'Energy', 'Entropy', 'InterquartileRange', 'Kurtosis', 'Maximum', 'Mean', 'MeanAbsoluteDeviation', 'Median', 'Minimum', 'Range', 'RobustMeanAbsoluteDeviation', 'RootMeanSquared', 'Skewness', 'TotalEnergy', 'Uniformity', 'Variance']
